In [3]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.svm import OneClassSVM
import random
import time
import pandas as pd

plt.rcParams['axes.unicode_minus'] = False

# 트랜잭션 데이터 샘플을 생성하는 함수
def generate_sample_data(num_samples=1000, anomaly_ratio=0.05):    
    data = []    
    current_time = int(time.time() * 1000)  
    # 현재 시간을 기준으로 Epoch 밀리초로 변환    
    for i in range(num_samples):
        # 정상적인 응답 시간: 100ms ~ 2000ms 사이
        normal_duration = random.randint(100, 2000)
        # 이상적인 응답 시간: 3000ms 이상 (일부는 매우 큰 값으로 설정)        
        anomaly_duration = random.randint(3000, 10000)        
        # 임의로 anomaly_ratio의 비율만큼 이상치 데이터 생성
        
        if random.random() < anomaly_ratio:            
            duration = anomaly_duration        
        else:            
            duration = normal_duration     
            
        # startEpochMillis와 endEpochMillis 생성
        start_epoch = current_time + random.randint(0, 10000)  
        # 최근 10초 사이에 시작한 트랜잭션
        end_epoch = start_epoch + duration
        # 샘플 트랜잭션 데이터 생성        
        transaction = {            
            "traceId": f"trace_{i}",            
            "endPoint": f"/service/endpoint_{random.randint(1, 10)}",            
            "serviceName": f"service_{random.randint(1, 5)}",            
            "statusCode": 200,            
            "startEpochMillis": start_epoch,            
            "endEpochMillis": end_epoch,            
            "duration": duration,            
            "startDateTime": pd.to_datetime(start_epoch, unit='ms').isoformat()        
        }        
        
        data.append(transaction)    
        
    return pd.DataFrame(data)  

    # DataFrame으로 반환# OCSVM을 통한 이상치 탐지
def detect_anomalies(response_times, threshold=1000):    
    # 전체 크기와 동일한 결과 배열을 먼저 1로 초기화    
    predictions = np.ones_like(response_times)    
    
    # Threshold 이상인 값만 One-Class SVM을 통해 이상치 탐지    
    mask = response_times > threshold    
    response_times_high = response_times[mask]   
    
    # 데이터를 Numpy 배열로 변환    
    response_times_log = np.log1p(response_times_high).reshape(-1, 1)    
    # OCSVM 모델 생성    
    ocsvm = OneClassSVM(kernel='rbf', gamma=0.00005, nu=0.05)  # nu: 이상치 비율    
    ocsvm.fit(response_times_log)    

    # 예측 (1은 정상, -1은 이상치)    
    predictions[mask] = ocsvm.predict(response_times_log)    
    
    print(predictions)    
    
    return predictions

# 메인 로직
if __name__ == "__main__":    
    # 1. 샘플 데이터 생성    
    api_data = generate_sample_data(num_samples=1000, anomaly_ratio=0.05)    

    # 2. 응답 시간 가져오기    
    response_times = api_data['duration'].values    

    # 3. 이상치 탐지    
    anomalies = detect_anomalies(response_times)    

    # 4. 결과 출력    
    # for i, (time, anomaly) in enumerate(zip(response_times, anomalies)):    
    #     print(    
    #         f"Trace ID: {api_data[i]['traceId']}, Response Time: {time} ms, Anomaly: {'Yes' if anomaly == -1 else 'No'}")    
    
    # 5. Scatter Plot으로 시각화    
    plt.figure(figsize=(10, 6))    

    # 정상 데이터: anomaly == 1    
    normal_data = response_times[anomalies == 1]    
    normal_index = np.where(anomalies == 1)[0]    

    # 이상치 데이터: anomaly == -1    
    anomaly_data = response_times[anomalies == -1]    
    anomaly_index = np.where(anomalies == -1)[0]    

    # 정상 데이터 scatter    
    plt.scatter(normal_index, normal_data, label='Normal', color='blue', alpha=0.5)    

    # 이상치 데이터 scatter    
    plt.scatter(anomaly_index, anomaly_data, label='Anomaly', color='red', alpha=0.5)    

    # 그래프 꾸미기    
    plt.title('OCSVM Anomaly Detection on Response Times')    
    plt.xlabel('Transaction Index')    
    plt.ylabel('Response Time (ms)')    
    plt.legend()    
    plt.show()